In [4]:
data_path = '/mlbio_scratch/anagupta/xenium_preprocessed/10xgenomics_alzheimers_disease_mouse_data.csv'
embeddings_path = '/mlbio_scratch/wen2/scMAE/xenium_preprocessed_indomain'
cell_images_path = '/mlbio_scratch/anagupta/luna/data/cell_images'

import pandas as pd
#  read and shuffle the data
data = pd.read_csv(data_path)
data = data.sample(frac=1).reset_index(drop=True)

# print data
print(data.head())

directories = [
    "TgCRND8_2_5",
    "TgCRND8_5_7",
    "TgCRND8_17_9",
    "wildtype_2_5",
    "wildtype_5_7",
    "wildtype_13_4"
    ]

# 'TgCRND8_2_5', 'TgCRND8_5_7', 
# # get data by donor
# donor_data = data[data['donor'] == 'donor_1']

# split the data into train and test
train_data = data[(data['donor'] == 'wildtype_2_5') | (data['donor'] == 'wildtype_5_7')]
test_data = data[(data['donor'] == 'wildtype_13_4') | (data['donor'] == 'TgCRND8_2_5')]

# save the data chnage the name of the file
# test_data_path = '/mlbio_scratch/anagupta/luna/data/train_test_split_3/test_data.csv'
# train_data_path = '/mlbio_scratch/anagupta/luna/data/train_test_split_3/train_data.csv'
# test_data.to_csv(test_data_path, index=False)
# train_data.to_csv(train_data_path, index=False)

# print the first 5 rows of the data
# print(test_data.head())







      cell_id      coord_X      coord_Y  transcript_counts  \
0  ecgeihlb-1  4920.008350  3070.302100                315   
1  bckemcbn-1   711.996451   260.153877                260   
2  maamjgbo-1  4427.954639  1957.559546                223   
3  hbgkbmba-1  2299.824622  4209.156836                110   
4  emfjhnfg-1  1997.733044  1946.090771                 99   

   control_probe_counts  control_codeword_counts  unassigned_codeword_counts  \
0                     0                        0                           0   
1                     0                        0                           0   
2                     0                        0                           0   
3                     0                        0                           0   
4                     0                        0                           0   

   total_counts   cell_area  nucleus_area  ...  Unc13c Vat1l Vcan  Vim  Vip  \
0           315  643.205625     35.357344  ...     0.0   0.0  0.0  

In [ ]:
data_path = '/mlbio_scratch/anagupta/xenium_preprocessed/10xgenomics_alzheimers_disease_mouse_data.csv'
cell_image_embeddings_path = '/mlbio_scratch/wen2/scMAE/xenium_preprocessed_indomain'
cell_images_path = '/mlbio_scratch/anagupta/luna/data/cell_images'

output_path = '/mlbio_scratch/anagupta/luna/data/train_test_split_1'

import pandas as pd
#  read and shuffle the data
data = pd.read_csv(data_path)
data = data.sample(frac=1).reset_index(drop=True)

# print data
print(data.head())

donors = {
    "TgCRND8_2_5",
    "TgCRND8_5_7",
    "TgCRND8_17_9",
    "wildtype_2_5",
    "wildtype_5_7",
    "wildtype_13_4"
}

test_donors = {
    "TgCRND8_5_7"
}

train_donors = donors - test_donors
print(train_donors)

# split the data into train and test
test_data = data[data['donor'].isin(test_donors)]
train_data = data[data['donor'].isin(train_donors)]

test_ids = test_data['cell_id'].values
train_ids = train_data['cell_id'].values

import os
import torch
import concurrent.futures

pt_files = os.listdir(cell_image_embeddings_path)
file_paths = [os.path.join(cell_image_embeddings_path, file) for file in pt_files]

def load_if_match(file_path):
    cell_id = os.path.basename(file_path).replace('.pt', '')
    if cell_id in test_ids:
        return ('test', cell_id, torch.load(file_path, map_location="cpu"))
    elif cell_id in train_ids:
        return ('train', cell_id, torch.load(file_path, map_location="cpu"))
    else:
        return None

train_embeddings = {}
test_embeddings = {}

with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    for result in executor.map(load_if_match, file_paths):
        if result is not None:
            split, cell_id, embedding = result
            if split == 'train':
                train_embeddings[cell_id] = embedding
            else:
                test_embeddings[cell_id] = embedding

# Optional: save to disk
os.makedirs(output_path, exist_ok=True)
torch.save(train_embeddings, os.path.join(output_path, "train_embeddings.pt"))
torch.save(test_embeddings, os.path.join(output_path, "test_embeddings.pt"))

      cell_id      coord_X      coord_Y  transcript_counts  \
0  bpijlola-1  4836.377563  1069.334656                 39   
1  jigjchno-1  4413.280518   474.072351                247   
2  laeeemgl-1  1353.263916  2398.471936                308   
3  fmlbccam-1  2088.723303  2148.975146                200   
4  lcichfgl-1  1249.231055  2180.982336                167   

   control_probe_counts  control_codeword_counts  unassigned_codeword_counts  \
0                     0                        0                           0   
1                     0                        0                           0   
2                     0                        0                           0   
3                     0                        0                           0   
4                     0                        0                           0   

   total_counts   cell_area  nucleus_area  ...  Unc13c Vat1l Vcan  Vim  Vip  \
0            39   47.730156     10.115000  ...     0.0   0.0  0.0  

In [2]:
data_path = '/mlbio_scratch/anagupta/xenium_preprocessed/10xgenomics_alzheimers_disease_mouse_data.csv'
cell_image_embeddings_path = '/mlbio_scratch/wen2/scMAE/xenium_preprocessed_indomain'
cell_images_path = '/mlbio_scratch/anagupta/luna/data/cell_images'

output_path = '/mlbio_scratch/anagupta/luna/data/train_test_split_1'

import pandas as pd
#  read and shuffle the data
data = pd.read_csv(data_path)
data = data.sample(frac=1).reset_index(drop=True)

import os
# save shuffled data
data.to_csv(os.path.join('/mlbio_scratch/anagupta/xenium_preprocessed', '10xgenomics_alzheimers_disease_mouse_data_shuffled.csv.csv'), index=False)

In [ ]:
import os
import torch
import pandas as pd
import concurrent.futures

cell_image_embeddings_path = '/mlbio_scratch/wen2/scMAE/xenium_preprocessed_indomain'
output_path = '/mlbio_scratch/anagupta/xenium_preprocessed'

# Get all .pt file paths
pt_files = os.listdir(cell_image_embeddings_path)
file_paths = [os.path.join(cell_image_embeddings_path, f) for f in pt_files]

# Load all .pt files (key = cell_id, value = tensor)
def load_embedding(file_path):
    cell_id = os.path.basename(file_path).replace('.pt', '')
    try:
        tensor = torch.load(file_path, map_location="cpu")
        return cell_id, tensor
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

# Load using multithreading
all_embeddings = {}

with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    for result in executor.map(load_embedding, file_paths):
        if result is not None:
            cell_id, tensor = result
            all_embeddings[cell_id] = tensor

print(f"Loaded {len(all_embeddings)} embeddings.")

# Save to disk
os.makedirs(output_path, exist_ok=True)
torch.save(all_embeddings, os.path.join(output_path, 'all_embeddings.pt'))
print(f"Saved {len(all_embeddings)} embeddings to {output_path}")

In [ ]:
import shutil
import torch
import pandas as pd

def prepare_data(data_path, embedding_file = None, cell_images_path = None, slice_images_path = None, output_path = None):
    data = pd.read_csv(data_path)

    test_donors = {"TgCRND8_5_7"}
    train_donors = set(data['donor'].unique()) - test_donors

    test_data = data[data['donor'].isin(test_donors)]
    train_data = data[data['donor'].isin(train_donors)]
    
    if embedding_file is not None:
        embeddings = torch.load(embedding_file)

        train_embeddings = {cid: embeddings[cid] for cid in train_data['cell_id'] if cid in embeddings}
        test_embeddings = {cid: embeddings[cid] for cid in test_data['cell_id'] if cid in embeddings}

        os.makedirs(output_path, exist_ok=True)
        torch.save(train_embeddings, os.path.join(output_path, "train_embeddings.pt"))
        print(f"Saved {len(train_embeddings)} train embeddings to {output_path}")
        torch.save(test_embeddings, os.path.join(output_path, "test_embeddings.pt"))
        print(f"Saved {len(test_embeddings)} test embeddings to {output_path}")
        
    if cell_images_path is not None:
        cell_images_tar = os.listdir(cell_images_path)
        
        os.makedirs(output_path, exist_ok=True)
        for cell_image_tar in cell_images_tar:
            donor = cell_image_tar.removesuffix('_cell_images.tar')
            if donor in train_donors:
                shutil.copy(os.path.join(cell_images_path, cell_image_tar), os.path.join(output_path, "train_cell_images", cell_image_tar))
            else:
                shutil.copy(os.path.join(cell_images_path, cell_image_tar), os.path.join(output_path, "test_cell_images", cell_image_tar))

data_path = '/mlbio_scratch/anagupta/xenium_preprocessed/10xgenomics_alzheimers_disease_mouse_data_shuffled.csv'
embedding_file = '/mlbio_scratch/anagupta/xenium_preprocessed/all_embeddings.pt'
cell_images_path = '/mlbio_scratch/anagupta/xenium_preprocessed/cell_images'
output_path = '/mlbio_scratch/anagupta/luna/data/train_test_split_1'

prepare_data(data_path, embedding_file=embedding_file, output_path=output_path)

Saved 291527 train embeddings to /mlbio_scratch/anagupta/luna/data/train_test_split_1
Saved 58681 test embeddings to /mlbio_scratch/anagupta/luna/data/train_test_split_1


In [1]:
data_path='/mlbio_scratch/anagupta/luna/data/train_test_split_1/test_data.csv'

import pandas as pd
#  read and shuffle the data
test_data = pd.read_csv(data_path)
test_data = test_data.sample(frac=1).reset_index(drop=True)

# print the first 5 rows of the data
print(test_data.head())

# save the data chnage the name of the file
data_path2 = '/mlbio_scratch/anagupta/luna/data/train_test_split_1/test_data2.csv'
test_data.to_csv(data_path2, index=False)

# print the first 5 rows of the data
print(test_data.head())



      cell_id      coord_X      coord_Y  transcript_counts  \
0  kpkpilkh-1  1543.281665  1746.632544                142   
1  lhniofng-1  2801.931824  2421.564917                254   
2  lgockapn-1  3075.538440  2141.887732                216   
3  gaedkfke-1   654.632449  3172.390503                436   
4  jkfphhef-1  4490.126245   329.105302                 94   

   control_probe_counts  control_codeword_counts  unassigned_codeword_counts  \
0                     0                        0                           0   
1                     0                        0                           0   
2                     0                        0                           0   
3                     0                        0                           0   
4                     0                        0                           0   

   total_counts   cell_area  nucleus_area  ...  Unc13c Vat1l Vcan  Vim  Vip  \
0           142  391.098281     34.499375  ...     0.0   1.0  0.0  

In [ ]:
if cell_images_path is not None:
        os.makedirs(output_path, exist_ok=True)
        output_tar_path = os.path.join(output_path, "merged_cell_images.tar")

        with tarfile.open(output_tar_path, "w") as out_tar:
            cell_images_tar = os.listdir(cell_images_path)
            for tar_name in cell_images_tar:
                if tar_name.endswith(".tar"):
                    tar_path = os.path.join(cell_images_path, tar_name)
                    with tarfile.open(tar_path, "r") as in_tar:
                        for member in in_tar.getmembers():
                            if member.name.endswith(".npy"):
                                file_obj = in_tar.extractfile(member)
                                if file_obj is not None:
                                    # Optional: prefix member name with donor ID to avoid collisions
                                    donor = tar_name.removesuffix("_cell_images.tar")
                                    member.name = f"{donor}/{os.path.basename(member.name)}"
                                    out_tar.addfile(member, file_obj)
            print(f"✅ Merged all .tar files into {output_tar_path}")

In [ ]:
# if donor in train_donors:
        #     os.makedirs(output_path+"/train_cell_images", exist_ok=True)
        #     shutil.copy(os.path.join(cell_images_path, cell_image_tar), os.path.join(output_path+"/train_cell_images", cell_image_tar))
        # else:
        #     os.makedirs(output_path+"/test_cell_images", exist_ok=True)
        #     shutil.copy(os.path.join(cell_images_path, cell_image_tar), os.path.join(output_path+"/test_cell_images", cell_image_tar))

In [5]:
# load /mlbio_scratch/anagupta/luna/data/train_test_split_1/train_data.csv

import pandas as pd
# import polars as pd
#  read and shuffle the data
data_path = '/mlbio_scratch/anagupta/luna/data/train_test_split_1/train_data.csv'
train_data = pd.read_csv(data_path)
# reader = pd.read_csv(data_path, chunksize=100_000)
# train_data = pd.concat(reader)
# train_data = train_data.sample(frac=1).reset_index(drop=True)

# print the first 5 rows of the data
print(train_data.head())

      cell_id     coord_X     coord_Y  transcript_counts  \
0  aaabfoap-1  843.688242  608.298163                 83   
1  aaabonpk-1  851.089270  612.555530                156   
2  aaadnjke-1  833.557346  611.501331                204   
3  aaaeilha-1  840.111331  616.144528                 30   
4  aaaenhoh-1  824.709283  622.894101                225   

   control_probe_counts  control_codeword_counts  unassigned_codeword_counts  \
0                     0                        0                           0   
1                     0                        0                           0   
2                     0                        0                           0   
3                     0                        0                           0   
4                     0                        0                           0   

   total_counts   cell_area  nucleus_area  ...  Unc13c Vat1l Vcan  Vim  Vip  \
0            83   87.919219     31.790000  ...     0.0   5.0  0.0  0.0  0.0   


In [2]:
train_data['cell_id'].loc

cell_id
str
"""aaabfoap-1"""
"""aaabonpk-1"""
"""aaadnjke-1"""
"""aaaeilha-1"""
"""aaaenhoh-1"""
…
"""oiminpdh-1"""
"""oimjabgg-1"""
"""oimjaijc-1"""


In [1]:
# load /mlbio_scratch/anagupta/10xgenomics_alzheimers_disease_mouse_model.h5ad

import anndata as ad
# load the h5ad file
h5ad_file = '/mlbio_scratch/anagupta/10xgenomics_alzheimers_disease_mouse_model.h5ad'
adata = ad.read_h5ad(h5ad_file)

# print the first 5 rows of the data
print(adata)

AnnData object with n_obs × n_vars = 351714 × 347
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'x', 'y', 'assay_ontology_term_id', 'sex_ontology_term_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'donor_id', 'condition_id', 'tissue_type', 'library_key', 'assay', 'organism', 'sex', 'tissue', 'dataset', 'nicheformer_split', 'niche', 'region'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype'
    uns: 'nicheformer_version', 'schema_version', 'title'


In [2]:
dataframe = adata.to_df()
print(dataframe.head())

            ENSMUSG00000026090  ENSMUSG00000035722  ENSMUSG00000032281  \
cell_id                                                                  
aaaaomnd-1                 0.0                 0.0                 0.0   
aaabamjh-1                 0.0                 0.0                 0.0   
aaabncdp-1                 1.0                 0.0                 2.0   
aaacifle-1                 1.0                 0.0                 0.0   
aaadpmhm-1                 3.0                 0.0                 1.0   

            ENSMUSG00000035783  ENSMUSG00000000530  ENSMUSG00000036545  \
cell_id                                                                  
aaaaomnd-1                 1.0                 0.0                 1.0   
aaabamjh-1                 0.0                 0.0                 0.0   
aaabncdp-1                 0.0                 0.0                 0.0   
aaacifle-1                 0.0                 0.0                 0.0   
aaadpmhm-1                 0.0       

In [4]:
# convert csv into h5ad

import pandas as pd
import anndata as ad

csv_file = '/mlbio_scratch/anagupta/luna/data/train_test_split_1/train_data.csv'
h5ad_file = '/mlbio_scratch/anagupta/luna/data/train_test_split_1/train_data.h5ad'

# read the csv file
data = pd.read_csv(csv_file)

with open(h5ad_file, 'w') as f:
    f.write(data.to_csv(index=False))

# print the first 5 rows of the data
print(data.head())

KeyboardInterrupt: 

In [6]:
import tarfile
import torch
import numpy as np
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, Tuple

def load_single_npy_from_tar(tar_path: str, tarinfo_name: str) -> Tuple[str, torch.Tensor]:
    with tarfile.open(tar_path, 'r') as tar:
        tarinfo = tar.getmember(tarinfo_name)
        file_obj = tar.extractfile(tarinfo)
        if file_obj is not None:
            data = file_obj.read()
            np_array = np.load(BytesIO(data)).astype(np.float32)
            tensor = torch.from_numpy(np_array)
            cell_id = os.path.basename(tarinfo.name).replace('.npy', '')
            return cell_id, tensor
    return None

def load_tar_images_parallel(tar_path: str, max_workers: int = 8) -> Dict[str, torch.Tensor]:
    with tarfile.open(tar_path, 'r') as tar:
        npy_members = [m.name for m in tar if m.name.endswith('.npy')]

    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(load_single_npy_from_tar, tar_path, name) for name in npy_members]
        for future in futures:
            try:
                result = future.result()
                if result is not None:
                    cell_id, tensor = result
                    results[cell_id] = tensor
            except Exception as e:
                print(f"[ERROR] Failed to extract: {e}")

    return results


In [ ]:
all_images = {}
import os
tar_files = os.listdir('/mlbio_scratch/anagupta/luna/data/train_test_split_1/train_cell_images')
tar_files = [os.path.join('/mlbio_scratch/anagupta/luna/data/train_test_split_1/train_cell_images', tar_file) for tar_file in tar_files]
for tar_path in tar_files:
    print(f"[INFO] Extracting from {tar_path}")
    images = load_tar_images_parallel(tar_path, max_workers=8)
    all_images.update(images)
    print(f"[INFO] Finished {tar_path}, total images so far: {len(all_images)}")


[INFO] Extracting from /mlbio_scratch/anagupta/luna/data/train_test_split_1/train_cell_images/TgCRND8_17_9_cell_images.tar


In [2]:
all_images = {}

import os
import tarfile
import numpy as np
from io import BytesIO
import torch

# Traverse all .tar files and extract .npy images into a dict
for root, _, files in os.walk('/mlbio_scratch/anagupta/luna/data/train_test_split_1/train_cell_images'):
    for file in files:
        if file.endswith('.tar'):
            tar_file_path = os.path.join(root, file)
            with tarfile.open(tar_file_path, 'r') as tar:
                for tarinfo in tar:
                    if tarinfo.name.endswith('.npy'):
                        cell_id = os.path.basename(tarinfo.name).replace('.npy', '')
                        file_obj = tar.extractfile(tarinfo)
                        if file_obj is not None:
                            image_data = file_obj.read()
                            np_image = np.load(BytesIO(image_data)).astype(np.float32)
                            tensor_image = torch.from_numpy(np_image)
                            all_images[cell_id] = tensor_image

            print(f"[INFO] Finished extracting from {tar_file_path}, total images so far: {len(all_images)}")

print(f"[INFO] Finished loading {len(all_images)} images for split '{split}'.")

[INFO] Finished extracting from /mlbio_scratch/anagupta/luna/data/train_test_split_1/train_cell_images/TgCRND8_17_9_cell_images.tar, total images so far: 62267


KeyboardInterrupt: 